<a href="https://colab.research.google.com/github/sinamahdavi/aml-2025-mistake-detection/blob/sanam/notebooks/results_step2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comprehensive Model Analysis

This notebook performs complete analysis of all model variants (MLP, Transformer, LSTM) across different backbones (Omnivore, SlowFast) and splits (recordings, step).

## Includes:
- Model training (optional)
- Comprehensive evaluation and comparison
- Backbone comparison charts
- AUC and F1 visualizations
- Comparison tables (CSV + PNG)
- Error type analysis
- Metrics heatmaps


In [1]:
%cd /content
!rm -rf code

!git clone --recursive -b sanam https://github.com/sinamahdavi/aml-2025-mistake-detection.git code
%cd code

/content
Cloning into 'code'...
remote: Enumerating objects: 656, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 656 (delta 1), reused 0 (delta 0), pack-reused 622 (from 1)
Receiving objects: 100% (656/656), 350.75 KiB | 9.74 MiB/s, done.
Resolving deltas: 100% (421/421), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 15.86 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'
/content/code


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# Setup data and checkpoints
# TODO: Update paths below with your own Google Drive paths

import os
import json
os.chdir('/content/code')

# Create directories
!mkdir -p data/video/omnivore
!mkdir -p checkpoints

# Extract features (update path to your omnivore features zip)
print("Extracting features...")
zip_path = "/content/drive/MyDrive/AML/data/features/omnivore.zip"
if os.path.exists(zip_path):
    # Remove old files if they exist
    !rm -rf data/video/omnivore/*

    # Extract to data/video/omnivore
    result = !unzip -q "{zip_path}" -d data/video/ 2>&1
    if result:
        print("Extraction output:", "\n".join(result))

    # If files were extracted to data/video/omnivore/omnivore/, move them up
    if os.path.exists("data/video/omnivore/omnivore"):
        !mv data/video/omnivore/omnivore/* data/video/omnivore/ 2>/dev/null || true
        !rmdir data/video/omnivore/omnivore 2>/dev/null || true

    # Verify extraction
    if os.path.exists("data/video/omnivore"):
        npz_files = [f for f in os.listdir("data/video/omnivore") if f.endswith('.npz')]
        num_files = len(npz_files)
        print(f"✅ Extracted {num_files} feature files to data/video/omnivore/")

        # Check if we have the expected number of files (~384)
        if num_files < 300:
            print(f"⚠️  Warning: Expected ~384 files, found only {num_files}")
            print("   The zip file may be incomplete or extraction failed.")

        # Verify a few expected files exist
        expected_samples = ["5_11_360p.mp4_1s_1s.npz", "15_30_360p.mp4_1s_1s.npz", "26_46_360p.mp4_1s_1s.npz"]
        missing = [f for f in expected_samples if f not in npz_files]
        if missing:
            print(f"⚠️  Missing expected files: {missing[:3]}")
            print("   This may cause training errors. Check your zip file.")
    else:
        print("⚠️  Extraction may have failed. Check the zip structure.")
else:
    print("⚠️  Features zip not found. Please update the path.")
    print(f"   Expected: {zip_path}")

# Extract SlowFast features (update path to your slowfast features zip)
print("\nExtracting SlowFast features...")
slowfast_zip_path = "/content/drive/MyDrive/AML/data/features/slowfast.zip"
if os.path.exists(slowfast_zip_path):
    # Create directory if it doesn't exist
    !mkdir -p data/video/slowfast

    # Remove old files if they exist
    !rm -rf data/video/slowfast/*

    # Extract to data/video/slowfast
    result = !unzip -q "{slowfast_zip_path}" -d data/video/ 2>&1
    if result:
        print("Extraction output:", "\n".join(result))

    # If files were extracted to data/video/slowfast/slowfast/, move them up
    if os.path.exists("data/video/slowfast/slowfast"):
        !mv data/video/slowfast/slowfast/* data/video/slowfast/ 2>/dev/null || true
        !rmdir data/video/slowfast/slowfast 2>/dev/null || true

    # Verify extraction
    if os.path.exists("data/video/slowfast"):
        npz_files = [f for f in os.listdir("data/video/slowfast") if f.endswith('.npz')]
        num_files = len(npz_files)
        print(f"✅ Extracted {num_files} SlowFast feature files to data/video/slowfast/")

        # Check if we have the expected number of files (~384)
        if num_files < 300:
            print(f"⚠️  Warning: Expected ~384 files, found only {num_files}")
            print("   The zip file may be incomplete or extraction failed.")

        # Verify a few expected files exist
        expected_samples = ["5_11_360p.mp4_1s_1s.npz", "15_30_360p.mp4_1s_1s.npz", "26_46_360p.mp4_1s_1s.npz"]
        missing = [f for f in expected_samples if f not in npz_files]
        if missing:
            print(f"⚠️  Missing expected files: {missing[:3]}")
            print("   This may cause training errors. Check your zip file.")
    else:
        print("⚠️  Extraction may have failed. Check the zip structure.")
else:
    print("⚠️  SlowFast features zip not found. Please update the path.")
    print(f"   Expected: {slowfast_zip_path}")
    print("   Note: You need SlowFast features to train SlowFast models.")

# Extract checkpoints if needed (optional)
if os.path.exists("/content/drive/MyDrive/AML/code/error_recognition_best.zip"):
  !unzip -q "/content/drive/MyDrive/AML/code/error_recognition_best.zip" -d checkpoints/ 2>&1
  print("✅ Checkpoints extracted")

Extracting features...
✅ Extracted 384 feature files to data/video/omnivore/

Extracting SlowFast features...
✅ Extracted 384 SlowFast feature files to data/video/slowfast/
✅ Checkpoints extracted


In [5]:
# Install required packages
!pip install torcheval tabulate
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 7.5 MB/s eta 0:00:00


In [9]:
# Train MLP + Omnivore on recordings split
import os
os.chdir('/content/code')
!python train_er.py --variant MLP --backbone omnivore --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'omnivore', 'ckpt_directory': './checkpoints', 'split': 'recordings', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this

In [6]:
# Train MLP + Omnivore on step split
import os
os.chdir('/content/code')
!python train_er.py --variant MLP --backbone omnivore --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'omnivore', 'ckpt_directory': './checkpoints', 'split': 'step', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataL

In [10]:
# Train MLP + SlowFast on recordings split
import os
os.chdir('/content/code')
!python train_er.py --variant MLP --backbone slowfast --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'slowfast', 'ckpt_directory': './checkpoints', 'split': 'recordings', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this

In [7]:
# Train MLP + SlowFast on step split
import os
os.chdir('/content/code')
!python train_er.py --variant MLP --backbone slowfast --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'slowfast', 'ckpt_directory': './checkpoints', 'split': 'step', 'variant': 'MLP', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataL

### Train Transformer Models


In [8]:
# Train Transformer + Omnivore on recordings split
import os
os.chdir('/content/code')
!python train_er.py --variant Transformer --backbone omnivore --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'omnivore', 'ckpt_directory': './checkpoints', 'split': 'recordings', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than w

In [11]:
# Train Transformer + Omnivore on step split
import os
os.chdir('/content/code')
!python train_er.py --variant Transformer --backbone omnivore --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'omnivore', 'ckpt_directory': './checkpoints', 'split': 'step', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what th

In [12]:
# Train Transformer + SlowFast on recordings split
import os
os.chdir('/content/code')
!python train_er.py --variant Transformer --backbone slowfast --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'slowfast', 'ckpt_directory': './checkpoints', 'split': 'recordings', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than w

In [13]:
# Train Transformer + SlowFast on step split
import os
os.chdir('/content/code')
!python train_er.py --variant Transformer --backbone slowfast --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


-------------------------------------------------------------
Training step model and testing on step level
Train args: {'num_workers': 8, 'pin_memory': False, 'shuffle': True, 'batch_size': 8}
Test args: {'num_workers': 8, 'pin_memory': False, 'shuffle': False, 'batch_size': 1}
{'batch_size': 8, 'test_batch_size': 1, 'num_epochs': 10, 'lr': 0.001, 'weight_decay': 0.001, 'ckpt': None, 'seed': 42, 'backbone': 'slowfast', 'ckpt_directory': './checkpoints', 'split': 'step', 'variant': 'Transformer', 'model_name': None, 'task_name': 'error_recognition', 'error_category': None, 'modality': ['video'], 'device': None}
-------------------------------------------------------------
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what th

In [15]:
# Train LSTM + Omnivore on recordings split
import os
os.chdir('/content/code')
!python train_lstm.py --variant LSTM --backbone omnivore --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

Training LSTM model for Error Recognition (Step 2b)
Backbone: omnivore
Split: recordings
Learning Rate: 0.001
Epochs: 10
Device: cuda
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Train Epoch: 1, Progress: 496/497, Loss: 0.999778: 100% 497/497 [00:34<00:00, 14.38it/s]
val Progress: 681/86: 100% 86/86 [00:04<00:00, 18.75it/s]
----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.3986175115207373, 'recall': 0.7361702127659574, 'f1': 0.5171898355754858, 'accuracy': 0.5256975036710719, 'auc': np.float64(0.5704989981871958), 'pr_auc': tensor(0.3845)}
val Step Level Metrics: {'precision': 0.13043478260869565, 'recall': 0.42857142857142855, 'f1': 0.2, 'accuracy': 0.7209302325581395, 'auc': np.float64(0.5605786618444846), 'pr_auc': te

In [16]:
# Train LSTM + Omnivore on step split
import os
os.chdir('/content/code')
!python train_lstm.py --variant LSTM --backbone omnivore --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

Training LSTM model for Error Recognition (Step 2b)
Backbone: omnivore
Split: step
Learning Rate: 0.001
Epochs: 10
Device: cuda
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Train Epoch: 1, Progress: 468/469, Loss: 0.781782: 100% 469/469 [00:33<00:00, 13.91it/s]
val Progress: 774/97: 100% 97/97 [00:04<00:00, 19.89it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in tha

In [17]:
# Train LSTM + SlowFast on recordings split
import os
os.chdir('/content/code')
!python train_lstm.py --variant LSTM --backbone slowfast --split recordings --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3

Training LSTM model for Error Recognition (Step 2b)
Backbone: slowfast
Split: recordings
Learning Rate: 0.001
Epochs: 10
Device: cuda
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Train Epoch: 1, Progress: 496/497, Loss: 0.650354: 100% 497/497 [00:20<00:00, 24.40it/s]
val Progress: 681/86: 100% 86/86 [00:02<00:00, 40.60it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'accuracy': 0.654

In [18]:
# Train LSTM + SlowFast on step split
import os
os.chdir('/content/code')
!python train_lstm.py --variant LSTM --backbone slowfast --split step --batch_size 8 --num_epochs 10 --lr 1e-3 --weight_decay 1e-3


Training LSTM model for Error Recognition (Step 2b)
Backbone: slowfast
Split: step
Learning Rate: 0.001
Epochs: 10
Device: cuda
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Train Epoch: 1, Progress: 468/469, Loss: 1.066054: 100% 469/469 [00:19<00:00, 24.63it/s]
val Progress: 774/97: 100% 97/97 [00:02<00:00, 44.55it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in tha

## Comprehensive Analysis

After training, run this section to generate all analysis visualizations and tables.


In [19]:
# Setup and Imports
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
import warnings
warnings.filterwarnings('ignore')

os.chdir('/content/code')

# Import project modules
from base import fetch_model, test_er_model
from constants import Constants as const
from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn, step_sequence_collate_fn

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Imports successful!")


✅ Imports successful!


In [20]:
# Configuration
MODELS = [const.MLP_VARIANT, const.TRANSFORMER_VARIANT, const.LSTM_VARIANT]
BACKBONES = [const.OMNIVORE, const.SLOWFAST]
SPLITS = [const.RECORDINGS_SPLIT, const.STEP_SPLIT]
THRESHOLDS = {const.RECORDINGS_SPLIT: 0.4, const.STEP_SPLIT: 0.6}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Create results directory
os.makedirs("results/comprehensive_analysis", exist_ok=True)
os.makedirs("results/comprehensive_analysis/tables", exist_ok=True)
os.makedirs("results/comprehensive_analysis/charts", exist_ok=True)
os.makedirs("results/comprehensive_analysis/heatmaps", exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Models: {MODELS}")
print(f"Backbones: {BACKBONES}")
print(f"Splits: {SPLITS}")
print(f"Thresholds: {THRESHOLDS}")


Device: cuda
Models: ['MLP', 'Transformer', 'LSTM']
Backbones: ['omnivore', 'slowfast']
Splits: ['recordings', 'step']
Thresholds: {'recordings': 0.4, 'step': 0.6}


### Helper Functions


In [21]:
class EvalConfig:
    """Simple config class for evaluation."""
    def __init__(self, backbone="omnivore", variant="MLP", split="recordings", device="cuda"):
        self.backbone = backbone
        self.modality = const.VIDEO
        self.phase = "test"
        self.segment_length = 1
        self.segment_features_directory = "data/"
        self.ckpt_directory = ""
        self.split = split
        self.batch_size = 1
        self.test_batch_size = 1
        self.seed = 1000
        self.device = device
        self.variant = variant
        self.task_name = const.ERROR_RECOGNITION


def find_best_checkpoint(variant, backbone, split="recordings"):
    """Find the best checkpoint for a given variant and backbone."""
    official_patterns = [
        f"checkpoints/error_recognition_best/{variant}/{backbone}/*{split}*.pt",
        f"checkpoints/error_recognition_best/{variant}/{backbone}/*.pt",
    ]
    trained_patterns = [
        f"checkpoints/error_recognition/{variant}/{backbone}/*_best.pt",
        f"checkpoints/error_recognition/{variant}/{backbone}/*best*.pt",
        f"checkpoints/error_recognition/{variant}/{backbone}/*{split}*.pt",
        f"checkpoints/error_recognition/{variant}/{backbone}/*.pt"
    ]

    for pattern in official_patterns + trained_patterns:
        ckpts = glob.glob(pattern)
        if ckpts:
            best_ckpts = [c for c in ckpts if '_best' in c.lower() or 'best' in c.lower()]
            if best_ckpts:
                return sorted(best_ckpts, key=os.path.getmtime)[-1]
            return sorted(ckpts, key=os.path.getmtime)[-1]
    return None


def evaluate_model(variant, backbone, split, device="cuda", threshold=0.4):
    """Evaluate a model and return metrics."""
    ckpt_path = find_best_checkpoint(variant, backbone, split)
    if not ckpt_path or not os.path.exists(ckpt_path):
        print(f"⚠️  No checkpoint found for {variant} + {backbone} + {split}")
        return None

    print(f"📊 Evaluating {variant} + {backbone} on {split} split...")

    config = EvalConfig(backbone=backbone, variant=variant, split=split, device=device)
    model = fetch_model(config)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    test_dataset = CaptainCookStepDataset(config, const.TEST, split)
    if variant in [const.LSTM_VARIANT, const.GRU_VARIANT]:
        test_loader = DataLoader(test_dataset, batch_size=1, collate_fn=step_sequence_collate_fn)
    else:
        test_loader = DataLoader(test_dataset, batch_size=1, collate_fn=collate_fn)

    criterion = torch.nn.BCEWithLogitsLoss()
    test_losses, sub_step_metrics, step_metrics = test_er_model(
        model, test_loader, criterion, device,
        phase="test",
        step_normalization=True,
        sub_step_normalization=True,
        threshold=threshold
    )

    return {
        'variant': variant,
        'backbone': backbone,
        'split': split,
        'accuracy': step_metrics[const.ACCURACY] * 100,
        'precision': step_metrics[const.PRECISION] * 100,
        'recall': step_metrics[const.RECALL] * 100,
        'f1': step_metrics[const.F1] * 100,
        'auc': step_metrics[const.AUC] * 100
    }

print("✅ Helper functions defined!")


✅ Helper functions defined!


In [22]:
# Evaluate all combinations
all_results = []

for variant in MODELS:
    for backbone in BACKBONES:
        for split in SPLITS:
            threshold = THRESHOLDS[split]
            result = evaluate_model(variant, backbone, split, DEVICE, threshold)
            if result:
                all_results.append(result)

# Convert to DataFrame
df_all = pd.DataFrame(all_results)

# Save raw results
df_all.to_csv("results/comprehensive_analysis/all_results.csv", index=False)
print(f"\n✅ Evaluated {len(all_results)} model configurations")
print(f"Results saved to: results/comprehensive_analysis/all_results.csv")


📊 Evaluating MLP + omnivore on recordings split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 38340/671: 100%|██████████| 671/671 [00:05<00:00, 123.38it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3964945261528254, 'recall': 0.5688109780280797, 'f1': 0.46727266803505685, 'accuracy': 0.5735263432446531, 'auc': np.float64(0.5988330748775713), 'pr_auc': tensor(0.3673)}
test Step Level Metrics: {'precision': 0.4090909090909091, 'recall': 0.8589211618257261, 'f1': 0.5542168674698795, 'accuracy': 0.503725782414307, 'auc': np.float64(0.6302808067162018), 'pr_auc': tensor(0.4020)}
----------------------------------------------------------------
📊 Evaluating MLP + omnivore on step split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 42347/798: 100%|██████████| 798/798 [00:05<00:00, 143.87it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.4096162736939436, 'recall': 0.2989708115404083, 'f1': 0.3456549302643129, 'accuracy': 0.6831416629277163, 'auc': np.float64(0.6541560352028618), 'pr_auc': tensor(0.3187)}
test Step Level Metrics: {'precision': 0.6607142857142857, 'recall': 0.14859437751004015, 'f1': 0.24262295081967214, 'accuracy': 0.7105263157894737, 'auc': np.float64(0.7573902166041213), 'pr_auc': tensor(0.3638)}
----------------------------------------------------------------
📊 Evaluating MLP + slowfast on recordings split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


...

----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.6631944444444444, 'recall': 0.7670682730923695, 'f1': 0.7113594040968343, 'accuracy': 0.8057644110275689, 'auc': np.float64(0.8684428058317057), 'pr_auc': tensor(0.5814)}
test Step Level Metrics: {'precision': 0.7258064516129032, 'recall': 0.7228915662650602, 'f1': 0.7243460764587525, 'accuracy': 0.8283208020050126, 'auc': np.float64(0.8684428058317057), 'pr_auc': tensor(0.6111)}
----------------------------------------------------------------
📊 Evaluating LSTM + slowfast on recordings split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 671/671: 100%|██████████| 671/671 [00:05<00:00, 128.92it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.3591654247391952, 'recall': 1.0, 'f1': 0.5285087719298246, 'accuracy': 0.3591654247391952, 'auc': np.float64(0.5055727106050372), 'pr_auc': tensor(0.3592)}
test Step Level Metrics: {'precision': 0.3568215892053973, 'recall': 0.9875518672199171, 'f1': 0.5242290748898678, 'accuracy': 0.3561847988077496, 'auc': np.float64(0.5055727106050372), 'pr_auc': tensor(0.3569)}
----------------------------------------------------------------
📊 Evaluating LSTM + slowfast on step split...
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


test Progress: 798/798: 100%|██████████| 798/798 [00:05<00:00, 147.22it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.31203007518796994, 'recall': 1.0, 'f1': 0.47564469914040114, 'accuracy': 0.31203007518796994, 'auc': np.float64(0.5087929129999048), 'pr_auc': tensor(0.3120)}
test Step Level Metrics: {'precision': 0.3137755102040816, 'recall': 0.9879518072289156, 'f1': 0.4762826718296225, 'accuracy': 0.32205513784461154, 'auc': np.float64(0.5087929129999048), 'pr_auc': tensor(0.3138)}
----------------------------------------------------------------

✅ Evaluated 12 model configurations
Results saved to: results/comprehensive_analysis/all_results.csv


### 1. Comparison Tables (CSV + PNG)


In [23]:
def create_table_image(df, title, filename):
    """Create a PNG image of a table."""
    fig, ax = plt.subplots(figsize=(14, max(6, len(df) * 0.5 + 2)))
    ax.axis('tight')
    ax.axis('off')

    table_data = []
    for _, row in df.iterrows():
        table_data.append([
            row['variant'], row['backbone'],
            f"{row['accuracy']:.2f}", f"{row['precision']:.2f}",
            f"{row['recall']:.2f}", f"{row['f1']:.2f}", f"{row['auc']:.2f}"
        ])

    headers = ["Model", "Backbone", "Accuracy", "Precision", "Recall", "F1", "AUC"]
    table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 2)

    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')

    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')

    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✅ Saved: {filename}")


# Create tables for each split and overall
for split in SPLITS:
    df_split = df_all[df_all['split'] == split].copy()
    df_split = df_split.sort_values(['variant', 'backbone'])

    csv_path = f"results/comprehensive_analysis/tables/comparison_{split}.csv"
    df_split.to_csv(csv_path, index=False)
    print(f"✅ Saved CSV: {csv_path}")

    png_path = f"results/comprehensive_analysis/tables/comparison_{split}.png"
    create_table_image(df_split, f"Model Comparison - {split.upper()} Split", png_path)

# Overall table
df_all_sorted = df_all.sort_values(['split', 'variant', 'backbone'])
csv_path = "results/comprehensive_analysis/tables/comparison_all.csv"
df_all_sorted.to_csv(csv_path, index=False)
print(f"✅ Saved CSV: {csv_path}")

png_path = "results/comprehensive_analysis/tables/comparison_all.png"
create_table_image(df_all_sorted, "Model Comparison - All Splits", png_path)

print("\n✅ All comparison tables created!")


✅ Saved CSV: results/comprehensive_analysis/tables/comparison_recordings.csv
✅ Saved: results/comprehensive_analysis/tables/comparison_recordings.png
✅ Saved CSV: results/comprehensive_analysis/tables/comparison_step.csv
✅ Saved: results/comprehensive_analysis/tables/comparison_step.png
✅ Saved CSV: results/comprehensive_analysis/tables/comparison_all.csv
✅ Saved: results/comprehensive_analysis/tables/comparison_all.png

✅ All comparison tables created!


### 2. Backbone Comparison Charts


In [24]:
def plot_backbone_comparison(df, split, save_path):
    """Plot backbone comparison chart."""
    df_split = df[df['split'] == split].copy()
    models = df_split['variant'].unique()
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    x = np.arange(len(models))
    width = 0.35

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        omnivore_values = []
        slowfast_values = []

        for model in models:
            omni_val = df_split[(df_split['variant'] == model) & (df_split['backbone'] == 'omnivore')][metric].values
            slow_val = df_split[(df_split['variant'] == model) & (df_split['backbone'] == 'slowfast')][metric].values
            omnivore_values.append(omni_val[0] if len(omni_val) > 0 else 0)
            slowfast_values.append(slow_val[0] if len(slow_val) > 0 else 0)

        bars1 = ax.bar(x - width/2, omnivore_values, width, label='Omnivore', alpha=0.8)
        bars2 = ax.bar(x + width/2, slowfast_values, width, label='SlowFast', alpha=0.8)

        ax.set_xlabel('Model', fontsize=12)
        ax.set_ylabel(f'{metric.upper()} (%)', fontsize=12)
        ax.set_title(f'{metric.upper()} - {split.upper()} Split', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(models)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.1f}%',
                       ha='center', va='bottom', fontsize=9)

    fig.delaxes(axes[5])
    plt.suptitle(f'Backbone Comparison - {split.upper()} Split', fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


for split in SPLITS:
    save_path = f"results/comprehensive_analysis/charts/backbone_comparison_{split}.png"
    plot_backbone_comparison(df_all, split, save_path)

print("\n✅ All backbone comparison charts created!")


✅ Saved: results/comprehensive_analysis/charts/backbone_comparison_recordings.png
✅ Saved: results/comprehensive_analysis/charts/backbone_comparison_step.png

✅ All backbone comparison charts created!


In [25]:
def plot_auc_chart(df, split, save_path):
    """Plot AUC comparison chart."""
    df_split = df[df['split'] == split].copy()
    models = df_split['variant'].unique()
    x = np.arange(len(models))
    width = 0.35

    omnivore_auc = []
    slowfast_auc = []

    for model in models:
        omni_val = df_split[(df_split['variant'] == model) & (df_split['backbone'] == 'omnivore')]['auc'].values
        slow_val = df_split[(df_split['variant'] == model) & (df_split['backbone'] == 'slowfast')]['auc'].values
        omnivore_auc.append(omni_val[0] if len(omni_val) > 0 else 0)
        slowfast_auc.append(slow_val[0] if len(slow_val) > 0 else 0)

    fig, ax = plt.subplots(figsize=(12, 8))
    bars1 = ax.bar(x - width/2, omnivore_auc, width, label='Omnivore', alpha=0.8, color='#4CAF50')
    bars2 = ax.bar(x + width/2, slowfast_auc, width, label='SlowFast', alpha=0.8, color='#2196F3')

    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('AUC (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'AUC Comparison - {split.upper()} Split', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, max(max(omnivore_auc), max(slowfast_auc)) * 1.15])

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}%',
                   ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


for split in SPLITS:
    save_path = f"results/comprehensive_analysis/charts/auc_{split}.png"
    plot_auc_chart(df_all, split, save_path)

print("\n✅ All AUC charts created!")


✅ Saved: results/comprehensive_analysis/charts/auc_recordings.png
✅ Saved: results/comprehensive_analysis/charts/auc_step.png

✅ All AUC charts created!


### 4. F1 Charts


In [26]:
def plot_f1_chart(df, split, save_path):
    """Plot F1 comparison chart."""
    df_split = df[df['split'] == split].copy()
    models = df_split['variant'].unique()
    x = np.arange(len(models))
    width = 0.35

    omnivore_f1 = []
    slowfast_f1 = []

    for model in models:
        omni_val = df_split[(df_split['variant'] == model) & (df_split['backbone'] == 'omnivore')]['f1'].values
        slow_val = df_split[(df_split['variant'] == model) & (df_split['backbone'] == 'slowfast')]['f1'].values
        omnivore_f1.append(omni_val[0] if len(omni_val) > 0 else 0)
        slowfast_f1.append(slow_val[0] if len(slow_val) > 0 else 0)

    fig, ax = plt.subplots(figsize=(12, 8))
    bars1 = ax.bar(x - width/2, omnivore_f1, width, label='Omnivore', alpha=0.8, color='#FF9800')
    bars2 = ax.bar(x + width/2, slowfast_f1, width, label='SlowFast', alpha=0.8, color='#9C27B0')

    ax.set_xlabel('Model', fontsize=14, fontweight='bold')
    ax.set_ylabel('F1 Score (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'F1 Score Comparison - {split.upper()} Split', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=12)
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, max(max(omnivore_f1), max(slowfast_f1)) * 1.15])

    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}%',
                   ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


for split in SPLITS:
    save_path = f"results/comprehensive_analysis/charts/f1_{split}.png"
    plot_f1_chart(df_all, split, save_path)

print("\n✅ All F1 charts created!")


✅ Saved: results/comprehensive_analysis/charts/f1_recordings.png
✅ Saved: results/comprehensive_analysis/charts/f1_step.png

✅ All F1 charts created!


### 5. Metrics Heatmaps


In [27]:
def create_heatmap(df, split, save_path):
    """Create metrics heatmap."""
    if split is None:
        df_split = df.copy()
        title = "Metrics Heatmap - All Splits"
    else:
        df_split = df[df['split'] == split].copy()
        title = f"Metrics Heatmap - {split.upper()} Split"

    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    heatmap_data = []

    for variant in df_split['variant'].unique():
        for backbone in df_split['backbone'].unique():
            if split is None:
                for s in df_split['split'].unique():
                    subset = df_split[(df_split['variant'] == variant) &
                                     (df_split['backbone'] == backbone) &
                                     (df_split['split'] == s)]
                    if len(subset) > 0:
                        row_data = [subset[m].values[0] for m in metrics]
                        heatmap_data.append({
                            'Model': f"{variant}_{backbone}_{s}",
                            **{m: row_data[i] for i, m in enumerate(metrics)}
                        })
            else:
                subset = df_split[(df_split['variant'] == variant) & (df_split['backbone'] == backbone)]
                if len(subset) > 0:
                    row_data = [subset[m].values[0] for m in metrics]
                    heatmap_data.append({
                        'Model': f"{variant}_{backbone}",
                        **{m: row_data[i] for i, m in enumerate(metrics)}
                    })

    if not heatmap_data:
        print(f"⚠️  No data for heatmap: {save_path}")
        return

    df_heatmap = pd.DataFrame(heatmap_data)
    df_heatmap = df_heatmap.set_index('Model')

    plt.figure(figsize=(10, max(6, len(df_heatmap) * 0.6)))
    sns.heatmap(df_heatmap, annot=True, fmt='.2f', cmap='YlOrRd',
                cbar_kws={'label': 'Score (%)'}, linewidths=0.5, linecolor='gray')
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Metrics', fontsize=12, fontweight='bold')
    plt.ylabel('Model + Backbone', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {save_path}")


for split in SPLITS:
    save_path = f"results/comprehensive_analysis/heatmaps/heatmap_{split}.png"
    create_heatmap(df_all, split, save_path)

save_path = "results/comprehensive_analysis/heatmaps/heatmap_all.png"
create_heatmap(df_all, None, save_path)

print("\n✅ All heatmaps created!")


✅ Saved: results/comprehensive_analysis/heatmaps/heatmap_recordings.png
✅ Saved: results/comprehensive_analysis/heatmaps/heatmap_step.png
✅ Saved: results/comprehensive_analysis/heatmaps/heatmap_all.png

✅ All heatmaps created!


### 6. Error Type Analysis


In [29]:
# Run error type analysis for each model
print("Running error type analysis...\n")

for variant in MODELS:
    for backbone in BACKBONES:
        for split in SPLITS:
            threshold = THRESHOLDS[split]
            ckpt_path = find_best_checkpoint(variant, backbone, split)

            if ckpt_path and os.path.exists(ckpt_path):
                print(f"Analyzing {variant} + {backbone} on {split} split...")
                cmd = f'python -m core.evaluate_error_types --variant {variant} --backbone {backbone} --split {split} --ckpt "{ckpt_path}" --threshold {threshold} --save_csv'
                os.system(cmd)
                print()

print("✅ Error type analysis complete!")
print("Results saved to: results/error_type_analysis/")

# Create PNG images from CSV files
print("\nGenerating PNG images for error type analysis...")

def create_error_type_table_image(csv_path, png_path):
    """Create a PNG image from error type analysis CSV."""
    try:
        # Read CSV file
        with open(csv_path, 'r') as f:
            lines = f.readlines()

        # Find the "Per Error Type Metrics" section
        per_error_start = None
        for i, line in enumerate(lines):
            if "Per Error Type Metrics" in line:
                per_error_start = i + 1  # Skip header line
                break

        if per_error_start is None:
            print(f"⚠️  Could not find 'Per Error Type Metrics' section in {csv_path}")
            return

        # Parse the per error type metrics
        table_data = []
        for line in lines[per_error_start:]:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) >= 7:
                error_type = parts[0]
                count = parts[1]
                accuracy = parts[2]
                precision = parts[3]
                recall = parts[4]
                f1 = parts[5]
                auc = parts[6]
                table_data.append([error_type, count, accuracy, precision, recall, f1, auc])

        if not table_data:
            print(f"⚠️  No error type data found in {csv_path}")
            return

        # Create figure
        fig, ax = plt.subplots(figsize=(14, max(6, len(table_data) * 0.5 + 2)))
        ax.axis('tight')
        ax.axis('off')

        headers = ["Error Type", "Count", "Accuracy", "Precision", "Recall", "F1", "AUC"]
        table = ax.table(cellText=table_data, colLabels=headers, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 2)

        # Style header
        for i in range(len(headers)):
            table[(0, i)].set_facecolor('#4CAF50')
            table[(0, i)].set_text_props(weight='bold', color='white')

        # Style rows (alternating colors)
        for i in range(1, len(table_data) + 1):
            for j in range(len(headers)):
                if i % 2 == 0:
                    table[(i, j)].set_facecolor('#f0f0f0')

        # Get variant, backbone, split from filename
        filename = os.path.basename(csv_path)
        parts = filename.replace('_error_type_analysis.csv', '').split('_')
        if len(parts) >= 3:
            variant_name = parts[0]
            backbone_name = parts[1]
            split_name = parts[2]
            title = f"Error Type Analysis - {variant_name} + {backbone_name} ({split_name})"
        else:
            title = "Error Type Analysis"

        plt.title(title, fontsize=16, fontweight='bold', pad=20)
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f"✅ Saved PNG: {png_path}")
    except Exception as e:
        print(f"⚠️  Error creating PNG for {csv_path}: {e}")

# Create PNG directory
os.makedirs("results/error_type_analysis/png", exist_ok=True)

# Generate PNG for each CSV file
for variant in MODELS:
    for backbone in BACKBONES:
        for split in SPLITS:
            csv_filename = f"{variant}_{backbone}_{split}_error_type_analysis.csv"
            csv_path = f"results/error_type_analysis/{csv_filename}"
            png_path = f"results/error_type_analysis/png/{csv_filename.replace('.csv', '.png')}"

            if os.path.exists(csv_path):
                create_error_type_table_image(csv_path, png_path)
            else:
                print(f"⚠️  CSV file not found: {csv_path}")

print("\n✅ All error type analysis PNG images created!")
print("PNG files saved to: results/error_type_analysis/png/")

Running error type analysis...

Analyzing MLP + omnivore on recordings split...

Analyzing MLP + omnivore on step split...

Analyzing MLP + slowfast on recordings split...

Analyzing MLP + slowfast on step split...

Analyzing Transformer + omnivore on recordings split...

Analyzing Transformer + omnivore on step split...

Analyzing Transformer + slowfast on recordings split...

Analyzing Transformer + slowfast on step split...

Analyzing LSTM + omnivore on recordings split...

Analyzing LSTM + omnivore on step split...

Analyzing LSTM + slowfast on recordings split...

Analyzing LSTM + slowfast on step split...

✅ Error type analysis complete!
Results saved to: results/error_type_analysis/

Generating PNG images for error type analysis...
✅ Saved PNG: results/error_type_analysis/png/MLP_omnivore_recordings_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/MLP_omnivore_step_error_type_analysis.png
✅ Saved PNG: results/error_type_analysis/png/MLP_slowfast_recordings_er